<a href="https://colab.research.google.com/github/fidlarsyn/grokking-Deep-Learning-Chapter-11-to-15/blob/main/13_Introducing_Automatic_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **What is a Deep Learning Framework?**

*Framework deep learning* (seperti PyTorch dan TensorFlow) dibuat untuk mengurangi kompleksitas kode, mempercepat pengembangan, mengurangi risiko eror, dan mengoptimalkan performa eksekusi (seperti penggunaan GPU). Fungsi paling vital dari sebuah *framework* adalah menyediakan fitur **automatic backpropagation (autograd)** dan **automatic optimization**. Dengan fitur ini, pengguna hanya perlu menuliskan logika *forward propagation*, sedangkan kalkulasi turunan (*gradients*) dan pembaruan bobot dilakukan secara otomatis.

# **Introduction to Tensors**

Tensor adalah bentuk abstraksi umum dari array bertingkat. Skalar adalah tensor 0D, vektor adalah tensor 1D, matriks adalah tensor 2D, dan array dimensi lebih tinggi disebut tensor $n$-D. Kelas Tensor dibentuk sebagai fondasi dasar framework untuk membungkus data NumPy sekaligus menyimpan informasi grafis komputasi.

# **Introduction to Automatic Gradient Computation (Autograd)**

Pada bab-bab sebelumnya, *backpropagation* dihitung secara manual. Autograd mengotomatiskan proses ini dengan membangun **Dynamic Computation Graph** selama *forward propagation*. Setiap objek `Tensor` menyimpan atribut `creators` (tensor penciptanya) dan `creation_op` (operasi matematikanya). Saat `.backward()` dipanggil, tensor akan merambatkan gradien mundur secara rekursif ke seluruh tensor pembuatnya.

# **A Quick Checkpoint**

Graf komputasi yang dibangun di atas objek Tensor mirip dengan alur memori pada RNN. Bedanya, *framework autograd* menyediakannya dalam bentuk antarmuka (*interface*) yang bersih sehingga pembaca tidak perlu lagi menulis logika *backprop* manual yang rumit.

# **Tensors That Are Used Multiple Times &amp; Upgrading Autograd**

Masalah terjadi jika satu tensor digunakan beberapa kali dalam *forward propagation* (misalnya matriks bobot yang dipakai berulang). Jika gradien langsung ditimpa, nilai gradien dari jalur lain akan hilang. Solusinya adalah meng-upgrad autograd agar **mengakumulasikan gradien** (`self.grad += grad`) dan menambahkan pengecekan `all_children_grads_accounted_for()`, sehingga tensor hanya merambatkan gradien ke orang tuanya setelah menerima seluruh gradien dari semua *children*\-nya.

In [ ]:
import numpy as np

class Tensor(object):
    def __init__(self, data, autograd=False, creators=None, creation_op=None, id=None):
        self.data = np.array(data)
        self.creators = creators
        self.creation_op = creation_op
        self.grad = None
        self.autograd = autograd
        self.children = {\}

        if id is None:
            id = np.random.randint(0, 100000)
        self.id = id

        if creators is not None:
            for c in creators:
                if self.id not in c.children:
                    c.children[self.id] = 1
                else:
                    c.children[self.id] += 1

    def all_children_grads_accounted_for(self):
        for ch, count in self.children.items():
            if count != 0:
                return False
        return True

    def backward(self, grad=None, grad_origin=None):
        if self.autograd:
            if grad is None:
                grad = Tensor(np.ones_like(self.data))

            if grad_origin is not None:
                if self.children[grad_origin.id] == 0:
                    raise Exception("cannot backprop more than once")
                else:
                    self.children[grad_origin.id] -= 1

            if self.grad is None:
                self.grad = grad
            else:
                self.grad += grad

            if self.creators is not None and (self.all_children_grads_accounted_for() or grad_origin is None):
                if self.creation_op == "add":
                    self.creators.backward(self.grad, self)
                    self.creators[37].backward(self.grad, self)

    def __add__(self, other):
        if self.autograd and other.autograd:
            return Tensor(self.data + other.data,
                          autograd=True,
                          creators=[self, other],
                          creation_op="add")
        return Tensor(self.data + other.data)

    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

# **How Does Addition Backpropagation Work? &amp; Adding Support for Negation and Additional Functions**

Setiap operasi matematika pada `Tensor` (seperti penjumlahan `__add__`, negasi `__neg__`, pengurangan `__sub__`, perkalian `__mul__`, penjumlahan dimensi `sum`, duplikasi dimensi `expand`, transpose `transpose`, dan perkalian matriks `mm`) mendaftarkan dirinya ke graf komputasi. Pada metode `.backward()`, setiap operasi menjalankan rumus turunan spesifiknya untuk merambatkan gradien mundur.

# **Using Autograd to Train a Neural Network &amp; Adding Automatic Optimization**

Menggunakan autograd menyederhanakan pelatihan jaringan saraf. Pengguna tidak perlu menyimpan variabel sementara atau menghitung delta manual. Dengan menambahkan kelas optimasi `SGD` (Stochastic Gradient Descent), pembaruan bobot (`optim.step()`) dan pembersihan gradien (`optim.zero()`) dapat dilakukan secara otomatis hanya dalam satu panggil fungsi.

# **Adding Support for Layer Types, Layers That Contain Layers, and Loss Functions**

Abstraksi kelas `Layer` menyatukan parameter bobot dan fungsi *forward propagation*.
* **Linear**: Layer linier yang menyimpan matriks `weight` dan `bias`.
* **Sequential**: Layer kontainer yang menghubungkan barisan layer secara berurutan.
* **MSELoss / CrossEntropyLoss**: Layer pengukur fungsi kerugian yang menghitung eror tanpa memiliki parameter bobot.

In [ ]:
class SGD(object):
    def __init__(self, parameters, alpha=0.1):
        self.parameters = parameters
        self.alpha = alpha

    def zero(self):
        for p in self.parameters:
            p.grad.data *= 0

    def step(self, zero=True):
        for p in self.parameters:
            p.data -= p.grad.data * self.alpha
            if zero:
                p.grad.data *= 0

class Layer(object):
    def __init__(self):
        self.parameters = list()

    def get_parameters(self):
        return self.parameters

class Linear(Layer):
    def __init__(self, n_inputs, n_outputs):
        super().__init__()
        W = np.random.randn(n_inputs, n_outputs) * np.sqrt(2.0 / n_inputs)
        self.weight = Tensor(W, autograd=True)
        self.bias = Tensor(np.zeros(n_outputs), autograd=True)
        self.parameters.append(self.weight)
        self.parameters.append(self.bias)

    def forward(self, input):
        return input.mm(self.weight) + self.bias.expand(0, len(input.data))

class Sequential(Layer):
    def __init__(self, layers=list()):
        super().__init__()
        self.layers = layers

    def add(self, layer):
        self.layers.append(layer)

    def forward(self, input):
        for layer in self.layers:
            input = layer.forward(input)
        return input

    def get_parameters(self):
        params = list()
        for l in self.layers:
            params += l.get_parameters()
        return params

class MSELoss(Layer):
    def __init__(self):
        super().__init__()

    def forward(self, pred, target):
        return ((pred - target) * (pred - target)).sum(0)

In [ ]:
import numpy as np

np.random.seed(0)

# Data Latih Biner Sederhana
data = Tensor(np.array([,[37]]), autograd=True)
target = Tensor(np.array([,[37],,[37]]), autograd=True)

# Membuat Model Sequential dengan 2 Layer Linier
model = Sequential([Linear(2, 3), Linear(3, 1)])
criterion = MSELoss()
optim = SGD(parameters=model.get_parameters(), alpha=0.05)

# Training Loop Menggunakan Otomatisasi Framework
for i in range(10):
    pred = model.forward(data)
    loss = criterion.forward(pred, target)
    loss.backward(Tensor(np.ones_like(loss.data)))
    optim.step()
    print(f"Iterasi {i+1:02d\} | Loss: {loss\}")

# **Nonlinearity Layers, Embedding Layer, and Indexing in Autograd**

Menambahkan operasi aktivasi non-linier seperti `.sigmoid()` dan `.tanh()` ke dalam `Tensor` dan bungkus kelas `Layer`. Agar dapat membangun **Embedding Layer**, autograd ditambahkan dukungan operasi `index_select()`, yang memastikan gradien dipetakan kembali secara akurat ke baris indeks yang sesuai saat *backpropagation*.

# **The Recurrent Neural Network Layer &amp; Summary**

Menggabungkan layer-layer dasar (`Linear`, `Embedding`, `Softmax`/`CrossEntropy`) untuk membangun **Recurrent Layer (RNN)**. Mengembangkan *framework* dari nol memberikan pemahaman mendalam tentang cara kerja internal *framework* industri seperti **PyTorch**, yang menggunakan arsitektur *dynamic computation graph* dan API serupa.